# 3. Weather data

This notebook requests weather data from OpenMeteo API for one point within each German Bundesland

Set up and installation

In [1]:
# !pip install -qqq openmeteo-requests
# !pip install -qqq requests-cache retry-requests numpy pandas

Test API call

In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"start_date": "2020-03-01",
	"end_date": "2020-03-15",
	"daily": ["weather_code", "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min"],
	"timezone": "Europe/Berlin",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(2).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(3).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["weather_code"] = daily_weather_code
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)


Coordinates: 52.5483283996582°N 13.407821655273438°E
Elevation: 38.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Daily data
                         date  weather_code  temperature_2m_mean  \
0  2020-03-01 00:00:00+00:00          53.0             6.762501   
1  2020-03-02 00:00:00+00:00          61.0             5.920834   
2  2020-03-03 00:00:00+00:00          51.0             4.362500   
3  2020-03-04 00:00:00+00:00          51.0             3.647917   
4  2020-03-05 00:00:00+00:00           3.0             4.400000   
5  2020-03-06 00:00:00+00:00          55.0             5.314583   
6  2020-03-07 00:00:00+00:00          53.0             4.520833   
7  2020-03-08 00:00:00+00:00           3.0             5.052084   
8  2020-03-09 00:00:00+00:00          51.0             6.995834   
9  2020-03-10 00:00:00+00:00          53.0             5.508333   
10 2020-03-11 00:00:00+00:00          55.0             9.591667   
11 2020-03-12 00:00:00+00:00         

It works!

Next step - scrape coordinates for central point in each Bundesland to use in API call

In [3]:
import duckdb
duckdb.query("INSTALL parquet;")
duckdb.query("LOAD parquet;")

states = duckdb.query(f"""
    SELECT DISTINCT state AS state
    FROM read_parquet('../Data/Processed/gbif_states.parquet')          
""").to_df()

In [4]:
states

,state
0,Bremen
1,Nordrhein-Westfalen
2,Rheinland-Pfalz
3,Sachsen
4,Mecklenburg-Vorpommern
5,Baden-Württemberg
6,Sachsen-Anhalt
7,Thüringen
8,Berlin
9,Brandenburg


In [7]:
import os
import pandas as pd
import requests
from bs4 import BeautifulSoup
from lat_lon_parser import parse

In [8]:
state_list = []
lats = []
lons = []

wiki_headers = {'User-Agent': 'Chrome/134.0.0.0'}

for state in states.state.tolist():
    url = "https://en.wikipedia.org/wiki/" + state
    try:
        response = requests.get(url, headers=wiki_headers)
        response.raise_for_status()
    except requests.exceptions.HTTPError:
        print(f"Page does not exist: {url}")
        print(f"Cannot return data for {city}\n")
        continue
    state_list.append(state)
    citysoup = BeautifulSoup(response.content, 'html.parser')
    lats.append(parse(citysoup.find(class_ = 'latitude').get_text()))
    lons.append(parse(citysoup.find(class_ = 'longitude').get_text()))

In [9]:
state_coords = pd.DataFrame({
    'state':state_list,
    'lat':lats,
    'lon':lons
})

state_coords

,state,lat,lon
0,Bremen,53.075833,8.807222
1,Nordrhein-Westfalen,51.466667,7.550000
2,Rheinland-Pfalz,49.913056,7.450000
3,Sachsen,51.026944,13.358889
4,Mecklenburg-Vorpommern,53.750000,12.500000
5,Baden-Württemberg,48.537778,9.041111
6,Sachsen-Anhalt,52.000000,11.700000
7,Thüringen,50.861111,11.051944
8,Berlin,52.520000,13.405000
9,Brandenburg,52.361944,13.008056


---
Now request weather data for states using the middle point of each state as coordinates, ~~from Jan 2004~~ from Oct 2023 to Dec 2024  
*(include 6 months before start of 2024 to calculate lagged weather variables)*

In [10]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": lats,
	"longitude": lons,
	"start_date": "2003-07-01",
	"end_date": "2024-12-31",
	"daily": ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "precipitation_sum", "wind_speed_10m_max"],
	"timezone": "Europe/Berlin",
}
responses = openmeteo.weather_api(url, params = params)

daily_data_list = []

# Process 3 locations
for response in responses:
	print(f"\nCoordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process daily data. The order of variables needs to be the same as requested.
	daily = response.Daily()
	daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
	daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
	daily_temperature_2m_min = daily.Variables(2).ValuesAsNumpy()
	daily_precipitation_sum = daily.Variables(3).ValuesAsNumpy()
	daily_wind_speed_10m_max = daily.Variables(4).ValuesAsNumpy()
	
	daily_data = {"date": pd.date_range(
		start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	)}
	
	daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
	daily_data["temperature_2m_max"] = daily_temperature_2m_max
	daily_data["temperature_2m_min"] = daily_temperature_2m_min
	daily_data["precipitation_sum"] = daily_precipitation_sum
	daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
	daily_data["lat"] = response.Latitude()
	daily_data["lon"] = response.Longitude()
	
	daily_dataframe = pd.DataFrame(data = daily_data)
	#print("\nDaily data\n", daily_dataframe)
	daily_data_list.append(daily_dataframe)
	
full_df = pd.concat(daily_data_list, ignore_index=True)
	


Coordinates: 53.04042053222656°N 8.830188751220703°E
Elevation: 30.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Coordinates: 51.49384689331055°N 7.500000476837158°E
Elevation: 220.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Coordinates: 49.876976013183594°N 7.513043403625488°E
Elevation: 355.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Coordinates: 51.00175476074219°N 13.36314868927002°E
Elevation: 296.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Coordinates: 53.743408203125°N 12.461538314819336°E
Elevation: 56.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Coordinates: 48.541297912597656°N 9.090909004211426°E
Elevation: 434.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Coordinates: 51.985939025878906°N 11.724770545959473°E
Elevation: 60.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'


In [11]:
full_df

,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,lat,lon
0,2003-07-01 00:00:00+00:00,17.709669,21.268002,15.467999,6.8,20.767975,53.040421,8.830189
1,2003-07-02 00:00:00+00:00,15.820084,18.418001,14.268000,7.7,21.068459,53.040421,8.830189
2,2003-07-03 00:00:00+00:00,15.299251,19.168001,12.518000,3.5,22.493519,53.040421,8.830189
3,2003-07-04 00:00:00+00:00,14.905499,16.818001,13.568000,7.8,22.896111,53.040421,8.830189
4,2003-07-05 00:00:00+00:00,15.901332,18.668001,14.168000,1.3,22.322901,53.040421,8.830189
...,...,...,...,...,...,...,...,...
125675,2024-12-27 00:00:00+00:00,1.664000,6.239000,-0.311000,0.0,6.379216,50.579964,9.079646
125676,2024-12-28 00:00:00+00:00,0.149417,3.539000,-1.761000,0.0,12.620554,50.579964,9.079646
125677,2024-12-29 00:00:00+00:00,-1.992250,-1.061000,-2.411000,0.0,13.878688,50.579964,9.079646
125678,2024-12-30 00:00:00+00:00,-1.946416,-1.161000,-2.561000,0.0,15.804164,50.579964,9.079646


Add states and remove lat/long (using spatial join, as the lat/lon coords returned from openmeteo are not exactly the same as in the state_coords dataframe)

In [12]:
import geopandas as gpd

# Convert full to GeoDataFrame
points_gdf = gpd.GeoDataFrame(
    full_df,
    geometry=gpd.points_from_xy(full_df.lon, full_df.lat),
    crs="EPSG:4326"
)

# Load and filter Germany states
states = gpd.read_file("../Data/Raw/ne_10m_admin_1_states_provinces/ne_10m_admin_1_states_provinces.shp")
states = states[states['admin'] == 'Germany'].to_crs("EPSG:4326")

# Spatial join
result = gpd.sjoin(
    points_gdf,
    states[['name', 'geometry']],
    how='left',
    predicate='within'
)

result = result.rename(columns={'name': 'state'}).drop(columns=['lat', 'lon', 'index_right', 'geometry'])

result

,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,state
0,2003-07-01 00:00:00+00:00,17.709669,21.268002,15.467999,6.8,20.767975,Bremen
1,2003-07-02 00:00:00+00:00,15.820084,18.418001,14.268000,7.7,21.068459,Bremen
2,2003-07-03 00:00:00+00:00,15.299251,19.168001,12.518000,3.5,22.493519,Bremen
3,2003-07-04 00:00:00+00:00,14.905499,16.818001,13.568000,7.8,22.896111,Bremen
4,2003-07-05 00:00:00+00:00,15.901332,18.668001,14.168000,1.3,22.322901,Bremen
...,...,...,...,...,...,...,...
125675,2024-12-27 00:00:00+00:00,1.664000,6.239000,-0.311000,0.0,6.379216,Hessen
125676,2024-12-28 00:00:00+00:00,0.149417,3.539000,-1.761000,0.0,12.620554,Hessen
125677,2024-12-29 00:00:00+00:00,-1.992250,-1.061000,-2.411000,0.0,13.878688,Hessen
125678,2024-12-30 00:00:00+00:00,-1.946416,-1.161000,-2.561000,0.0,15.804164,Hessen


Check for NAs and sense check

In [13]:
result.describe()

,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max
count,125680.000000,125680.000000,125680.000000,125680.000000,125680.000000
mean,10.034213,13.634168,6.535543,2.104869,17.607849
std,7.270553,8.127982,6.644057,3.898492,6.688680
min,-18.066999,-13.204500,-24.674999,0.000000,3.462369
25%,4.476563,7.230500,1.613000,0.000000,12.574260
50%,10.081124,13.700000,6.700000,0.300000,16.595179
75%,15.845584,19.925001,11.900000,2.600000,21.602999
max,31.122917,38.900002,24.463001,107.000000,64.716942


In [14]:
result.isna().sum()

date                   0
temperature_2m_mean    0
temperature_2m_max     0
temperature_2m_min     0
precipitation_sum      0
wind_speed_10m_max     0
state                  0
dtype: int64

In [15]:
result.shape

(125680, 7)

Save to parquet

In [16]:
result.to_parquet('../Data/Processed/weather_states.parquet', index=False)